<a href="https://colab.research.google.com/github/CodeHunterOfficial/ArabovMKDeep/blob/main/NLP-2026/Lecture_1/HuggingFace_Tokenizers.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Лекция: HuggingFace Tokenizers — практическое обучение и использование субсловных токенизаторов

## 1. Введение

В предыдущих лекциях мы изучили теоретические основы трёх основных алгоритмов субсловной токенизации: BPE, WordPiece и Unigram Language Model. Каждый из них имеет свои особенности и области применения. Однако теория без практики остаётся неполной. В реальных проектах по обработке естественного языка токенизаторы редко реализуются с нуля; вместо этого используются готовые библиотеки, которые предоставляют эффективные, протестированные и оптимизированные реализации.

Одной из самых популярных библиотек для работы с токенизаторами является **HuggingFace Tokenizers**. Она написана на Rust с привязками для Python и обеспечивает высокую скорость обучения и токенизации (до нескольких миллионов текстов в секунду). Библиотека поддерживает обучение BPE, WordPiece и Unigram, а также предоставляет доступ к множеству предобученных токенизаторов из экосистемы HuggingFace (BERT, GPT-2, RoBERTa, T5 и др.).

В этой лекции мы рассмотрим, как использовать HuggingFace Tokenizers для:
- обучения собственного токенизатора на заданном корпусе с выбором одного из трёх алгоритмов;
- настройки предобработки текста (нормализация, предварительная токенизация, декодирование);
- загрузки и применения предобученных токенизаторов;
- сравнения скорости и качества разных подходов.

Мы также уделим особое внимание **обучению токенизаторов для русского, татарского и таджикского языков**, поскольку эти языки имеют свои особенности алфавита, морфологии и орфографии. Приведём примеры кода на Python, которые можно адаптировать под собственные задачи.

## 2. Установка и базовые компоненты

Библиотека `tokenizers` устанавливается через pip:


In [ ]:
!pip install tokenizers


Основными строительными блоками являются:

- **Tokenizer** — контейнер, который связывает все компоненты (модель, пре-токенизатор, декодер, нормализатор, пост-процессор) и предоставляет методы `train`, `encode`, `decode`.
- **Модель** (`models`) — определяет алгоритм (BPE, WordPiece, Unigram) и хранит выученный словарь.
- **Пре-токенизатор** (`pre_tokenizers`) — разбивает входную строку на предварительные токены (например, по пробелам или на байты) перед обучением и инференсом.
- **Нормализатор** (`normalizers`) — приводит текст к единому виду (нижний регистр, удаление диакритики, нормализация Unicode).
- **Декодер** (`decoders`) — собирает токены обратно в строку.
- **Тренер** (`trainers`) — управляет процессом обучения модели, содержит гиперпараметры (размер словаря, минимальная частота, специальные токены).
- **Пост-процессор** (`post_processors`) — добавляет специальные токены (например, `[CLS]`, `[SEP]`) после токенизации.

## 3. Обучение BPE токенизатора

### 3.1. Краткое напоминание теории

BPE итеративно объединяет самые частые пары соседних токенов. Начальный словарь состоит из всех символов (или байтов). На каждом шаге выбирается пара $(a,b)$ с максимальной частотой $f(a,b)$, создаётся новый токен $ab$, и все вхождения этой пары заменяются на $ab$. Процесс продолжается, пока размер словаря не достигнет заданного значения.

### 3.2. Практика с HuggingFace Tokenizers

Для обучения BPE используем модель `models.BPE` и тренер `trainers.BpeTrainer`.

**Пример:** Обучим BPE на небольшом списке текстов.




In [ ]:
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Whitespace

# Инициализируем токенизатор с моделью BPE
tokenizer = Tokenizer(BPE(unk_token="[UNK]"))

# Устанавливаем пре-токенизатор на пробелы (можно использовать другие)
tokenizer.pre_tokenizer = Whitespace()

# Тренер: задаём размер словаря, специальные токены, минимальную частоту
trainer = BpeTrainer(vocab_size=5000, min_frequency=2, special_tokens=["[UNK]", "[CLS]", "[SEP]", "[PAD]", "[MASK]"])

# Обучаем на списке файлов или итераторе текстов
texts = [
    "low lower lowest",
    "low low low",
    "lowering the lowest",
    "The quick brown fox jumps over the lazy dog",
    "hello world"
]
tokenizer.train_from_iterator(texts, trainer=trainer)

# Сохраняем токенизатор
tokenizer.save("bpe_tokenizer.json")


**Пояснения:**
- `BpeTrainer` принимает `vocab_size` — целевой размер словаря (включая специальные токены).
- `min_frequency` — минимальное число вхождений пары, чтобы её можно было слить. Позволяет отсечь редкие пары.
- `special_tokens` — список служебных токенов, которые добавляются в словарь до обучения.
- `train_from_iterator` принимает итератор по текстам (или по батчам текстов). Для больших корпусов можно передавать список файлов.

После обучения можно использовать токенизатор:


In [ ]:
output = tokenizer.encode("lowering")
print(output.tokens)   # ['low', 'e', 'ring'] (примерно, зависит от словаря)
print(output.ids)      # числовые идентификаторы


### 3.3. Дополнительные параметры BPE

- `continuing_subword_prefix` — префикс для подслов, которые не являются первыми в слове. Например, `"##"`.
- `end_of_word_suffix` — суффикс для обозначения конца слова (обычно `"</w>"`).
- `byte_fallback` — если True, то неизвестные символы кодируются как байты (аналог Byte-level BPE), что делает токенизатор способным обрабатывать любые символы.

## 4. Обучение WordPiece токенизатора

### 4.1. Краткое напоминание теории

WordPiece также итеративно сливает пары, но критерий выбора пары — не просто частота, а отношение $f(a,b)/(f(a)f(b))$. Это соответствует максимизации прироста логарифмического правдоподобия корпуса в униграммной модели. При инференсе используется жадное сопоставление слева направо.

### 4.2. Практика с HuggingFace Tokenizers


In [ ]:
from tokenizers import Tokenizer
from tokenizers.models import WordPiece
from tokenizers.trainers import WordPieceTrainer
from tokenizers.pre_tokenizers import Whitespace

tokenizer = Tokenizer(WordPiece(unk_token="[UNK]"))
tokenizer.pre_tokenizer = Whitespace()

trainer = WordPieceTrainer(
    vocab_size=5000,
    min_frequency=2,
    special_tokens=["[UNK]", "[CLS]", "[SEP]", "[PAD]", "[MASK]"]
)

texts = [
    "low lower lowest",
    "low low low",
    "lowering the lowest",
    "The quick brown fox jumps over the lazy dog",
    "hello world"
]

tokenizer.train_from_iterator(texts, trainer=trainer)
tokenizer.save("wordpiece_tokenizer.json")


**Особенности:**
- `WordPieceTrainer` использует другую score-функцию, заложенную в реализации.
- Можно задать `continuing_subword_prefix="##"`, чтобы токены, продолжающие слово, помечались.
- Как и в BPE, поддерживается `min_frequency`.

### 4.3. Сравнение результатов

На одном и том же корпусе BPE и WordPiece могут дать разные словари. WordPiece часто выделяет редкие, но сильно связанные пары раньше, чем BPE. Например, в корпусе `"low lower lowest"` WordPiece сначала объединит `(s,t)`, так как score = 1, а BPE начнёт с `(l,o)`, имеющей частоту 3. Это иллюстрирует, как критерий влияет на итоговое разбиение.

## 5. Обучение Unigram токенизатора

### 5.1. Краткое напоминание теории

Unigram LM строит словарь путём удаления из избыточного набора всех подслов, а не слияния. На каждом шаге оцениваются вероятности подслов с помощью EM-алгоритма, затем удаляются наименее полезные подслова (с минимальной потерей правдоподобия). Процесс продолжается до достижения целевого размера словаря.

### 5.2. Практика с HuggingFace Tokenizers




In [ ]:
from tokenizers import Tokenizer
from tokenizers.models import Unigram
from tokenizers.trainers import UnigramTrainer
from tokenizers.pre_tokenizers import Whitespace

tokenizer = Tokenizer(Unigram())
tokenizer.pre_tokenizer = Whitespace()

trainer = UnigramTrainer(
    vocab_size=5000,
    min_frequency=2,
    special_tokens=["[UNK]", "[CLS]", "[SEP]", "[PAD]", "[MASK]"],
    unk_token="[UNK]"
)

texts = [
    "low lower lowest",
    "low low low",
    "lowering the lowest",
    "The quick brown fox jumps over the lazy dog",
    "hello world"
]

tokenizer.train_from_iterator(texts, trainer=trainer)
tokenizer.save("unigram_tokenizer.json")


**Параметры UnigramTrainer:**
- `vocab_size` — целевой размер.
- `min_frequency` — минимальная частота подслова для включения в начальный словарь.
- `max_piece_length` — максимальная длина подслова (по умолчанию 16).
- `n_sub_iterations` — количество итераций удаления (по умолчанию 2).
- `shuffle` — перемешивание корпуса перед обучением.
- `unk_token` — токен для неизвестных символов.

**Важно:** Unigram использует вероятностную сегментацию. При инференсе можно выбрать Viterbi (наиболее вероятное разбиение) или сэмплирование (используется для регуляризации в моделях типа T5). В HuggingFace Tokenizers по умолчанию используется Viterbi, но при желании можно настроить сэмплирование.

## 6. Использование предобученных токенизаторов

Библиотека HuggingFace предоставляет доступ к множеству предобученных токенизаторов через библиотеку `transformers`. Часто проще использовать готовый токенизатор, чем обучать собственный.

**Примеры загрузки предобученных токенизаторов:**




In [ ]:
from transformers import BertTokenizer, GPT2Tokenizer, T5Tokenizer

# BERT (WordPiece)
bert_tok = BertTokenizer.from_pretrained('bert-base-uncased')
tokens = bert_tok.tokenize("The quick brown fox")
print(tokens)  # ['the', 'quick', 'brown', 'fox'] (с префиксами ## для подслов)

# GPT-2 (Byte-level BPE)
gpt2_tok = GPT2Tokenizer.from_pretrained('gpt2')
tokens = gpt2_tok.tokenize("The quick brown fox")
print(tokens)  # ['The', ' quick', ' brown', ' fox'] (пробелы в токенах)

# T5 (Unigram/SentencePiece)
t5_tok = T5Tokenizer.from_pretrained('t5-small')
tokens = t5_tok.tokenize("The quick brown fox")
print(tokens)  # ['▁The', '▁quick', '▁brown', '▁fox'] (нижнее подчёркивание как маркер пробела)


Каждый предобученный токенизатор имеет свой словарь, правила нормализации и пост-обработки. Важно понимать, что выбор токенизатора влияет на вход модели, поэтому нельзя смешивать токенизаторы разных моделей без необходимости.

## 7. Обучение токенизатора для русского, татарского и таджикского языков

Теперь перейдём к практическому вопросу: как обучить токенизатор для трёх языков, имеющих различные особенности. Этот раздел будет полезен, если вы планируете создать многоязычную модель или отдельные модели для каждого языка.

### 7.1. Особенности языков

- **Русский язык** использует кириллицу (33 буквы). Морфология богатая: склонения, спряжения, приставки и суффиксы. Слова могут быть длинными, часты приставки и окончания.
- **Татарский язык** также использует кириллицу (в России) с дополнительными буквами (ә, ө, ү, җ, ң, һ). Агглютинативный строй: цепочки аффиксов, выражающих падеж, число, принадлежность и др.
- **Таджикский язык** официально использует кириллицу (в Таджикистане), но также встречается арабская графика (в Афганистане). Мы будем ориентироваться на кириллический вариант. Таджикский — иранский язык, также агглютинативный, с множеством аффиксов.

Все три языка имеют значительную степень словоизменения, поэтому субсловная токенизация особенно полезна: она позволяет выделять морфемы, что уменьшает размер словаря и улучшает обработку редких слов.

### 7.2. Подготовка корпуса

Для обучения токенизатора необходим достаточно большой текстовый корпус на всех трёх языках. Рекомендуется:
- Использовать смешанный корпус, содержащий тексты на русском, татарском и таджикском. Соотношение может быть равным или соответствовать ожидаемому распределению в приложении.
- Выполнить очистку текста: удалить лишние пробелы, HTML-теги, знаки препинания оставить как есть (они важны).
- Привести текст к нижнему регистру (опционально, но для этих языков обычно полезно, так как уменьшает словарь и упрощает нормализацию).
- Убедиться, что текст в кодировке UTF-8.

### 7.3. Выбор алгоритма и параметров

Для агглютинативных и флективных языков хорошо подходят:
- **BPE** — простой и быстрый, хорошо работает.
- **Unigram** — часто даёт более качественные словари за счёт глобальной оптимизации, но обучение медленнее.
- **WordPiece** — также допустим, но его преимущество перед BPE не так велико.

Для **учебных целей и наглядности** мы выберем **символьный BPE с пробельным пре-токенизатором** (`Whitespace`). Это даст читаемые подслова (например, `при` + `вет`), что облегчает понимание. Если в будущем потребуется обрабатывать произвольные символы (включая арабскую графику), можно перейти на Byte-level BPE, но это выходит за рамки данного примера.

Параметры:
- `vocab_size` = 50 000 (или 30 000 для экономии). Для демонстрационного крошечного корпуса уменьшите до 300.
- `min_frequency` = 2 (или 1, если корпус мал).
- Специальные токены: `[UNK]`, `[CLS]`, `[SEP]`, `[PAD]`, `[MASK]`.


### 7.4. Пример кода для обучения на трёх языках




In [ ]:
import os
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Whitespace

# Создаём папку и файлы с примерами
os.makedirs("corpus", exist_ok=True)
with open("corpus/russian_corpus.txt", "w", encoding="utf-8") as f:
    f.write("Привет, как дела?\nЭто пример русского текста для обучения токенизатора.\n")
with open("corpus/tatar_corpus.txt", "w", encoding="utf-8") as f:
    f.write("Сәлам, хәлләрең ничек?\nБу татар телендәге текст мисалы.\n")
with open("corpus/tajik_corpus.txt", "w", encoding="utf-8") as f:
    f.write("Салом, аҳвол чӣ хел?\nИн матни тоҷикӣ барои омӯзиши токенизатор.\n")

# Инициализируем символьный BPE с пробельным пре-токенизатором
tokenizer = Tokenizer(BPE(unk_token="[UNK]"))
tokenizer.pre_tokenizer = Whitespace()

trainer = BpeTrainer(
    vocab_size=300,          # для примера; для реального корпуса ставьте 50000
    min_frequency=1,
    special_tokens=["[UNK]", "[CLS]", "[SEP]", "[PAD]", "[MASK]"]
)

files = [
    "corpus/russian_corpus.txt",
    "corpus/tatar_corpus.txt",
    "corpus/tajik_corpus.txt"
]
tokenizer.train(files, trainer=trainer)
tokenizer.save("multilingual_bpe_tokenizer.json")
print("Токенизатор сохранён.")


**Пояснения:**
- `Whitespace()` разбивает текст по пробелам, после чего каждый токен обрабатывается как последовательность символов.
- Кириллические буквы (включая дополнительные татарские и таджикские) являются обычными символами и попадают в словарь.
- `vocab_size` уменьшен до 300 из-за маленького демонстрационного корпуса. При наличии большого корпуса используйте 50 000.

---

### 7.5. Использование SentencePiece как альтернативы

Если вы предпочитаете SentencePiece (например, для унификации с моделями T5 или ALBERT), можно обучить модель так:



In [ ]:
import os
import sentencepiece as spm

# Создаём папку и файлы с примерами
os.makedirs("corpus", exist_ok=True)
with open("corpus/russian.txt", "w", encoding="utf-8") as f:
    f.write("Привет, как дела?\nЭто пример русского текста для обучения токенизатора.\n")
with open("corpus/tatar.txt", "w", encoding="utf-8") as f:
    f.write("Сәлам, хәлләрең ничек?\nБу татар телендәге текст мисалы.\n")
with open("corpus/tajik.txt", "w", encoding="utf-8") as f:
    f.write("Салом, аҳвол чӣ хел?\nИн матни тоҷикӣ барои омӯзиши токенизатор.\n")

spm.SentencePieceTrainer.train(
    input=["corpus/russian.txt", "corpus/tatar.txt", "corpus/tajik.txt"],
    model_prefix="multilingual_spm",
    vocab_size=300,                  # уменьшено для примера; для реального корпуса 50000
    model_type="bpe",
    character_coverage=0.9995,
    byte_fallback=True,
    max_sentence_length=4192,
    input_sentence_size=10000000,
    shuffle_input_sentence=True,
    pad_id=0, unk_id=1, bos_id=2, eos_id=3
)

print("Обучение завершено.")


После обучения модель загружается и используется аналогично токенизатору HuggingFace.



In [ ]:
sp = spm.SentencePieceProcessor()
sp.load("multilingual_spm.model")
print(sp.encode("привет", out_type=str))



### 7.6. Обработка таджикского языка с арабской графикой

Если в корпусе встречается таджикский в арабской графике (или вы хотите поддержать её), можно:
- Использовать **Byte-level BPE** (пре-токенизатор `ByteLevel` и модель BPE с `byte_fallback=True`), который автоматически обрабатывает любые символы.
- Либо обучить отдельный токенизатор для арабской графики.
- В нашем примере используется только кириллическая графика, поэтому Byte-level не требуется.



### 7.7. Проверка результатов

После обучения полезно проверить, как токенизатор сегментирует слова на каждом языке. Пример:




In [ ]:
tokenizer = Tokenizer.from_file("multilingual_bpe_tokenizer.json")

for word in ["привет", "сәлам", "салом", "китап", "китоб"]:
    output = tokenizer.encode(word)
    print(f"{word} -> {output.tokens}")


Ожидается, что слова разбиваются на осмысленные подслова, например, `привет` может остаться целым, а `сәлам` может разбиться на `с` + `ә` + `лам`.

## 8. Сравнение скорости и качества

### 8.1. Скорость

Библиотека HuggingFace Tokenizers написана на Rust, что обеспечивает высокую скорость. По тестам разработчиков, она способна токенизировать до 1-2 миллионов текстов в секунду на одном ядре. Для сравнения, реализация на чистом Python обычно на порядок медленнее. При обучении на корпусах объёмом в миллиарды слов использование этой библиотеки практически безальтернативно.

### 8.2. Качество сегментации

Качество субсловной сегментации субъективно, но можно использовать некоторые метрики:
- **Средняя длина токена** (в символах или байтах) — отражает компактность.
- **Доля слов, разбитых на подслова** — показывает, насколько часто модель вынуждена дробить слова.
- **Покрытие словаря** — насколько хорошо словарь покрывает тестовый корпус.

На практике BPE и WordPiece дают похожие результаты, но WordPiece может быть чуть лучше для языков с богатой морфологией за счёт более обоснованного критерия. Unigram, благодаря вероятностной оптимизации, часто даёт более компактные словари при том же размере.

### 8.3. Таблица сравнения

| Критерий               | BPE              | WordPiece        | Unigram           |
|------------------------|------------------|------------------|-------------------|
| Скорость обучения      | Высокая          | Высокая          | Средняя (EM)      |
| Скорость инференса     | Очень высокая    | Очень высокая    | Высокая           |
| Глобальная оптимизация | Нет (жадно)      | Нет (жадно)      | Да (удаление)     |
| Вероятностная сегментация | Нет           | Нет              | Да                |
| Используется в моделях | GPT, RoBERTa     | BERT, DistilBERT | T5, ALBERT, XLNet |

## 9. Практические рекомендации

1. **Всегда добавляйте специальные токены** (`[UNK]`, `[CLS]`, `[SEP]`, `[PAD]`, `[MASK]`) с самого начала обучения. Если их не добавить, их придётся добавлять вручную после обучения, что может привести к несоответствию индексов.

2. **Выбирайте размер словаря на основе задачи.** Для английского языка обычно достаточно 30 000–50 000 токенов. Для многоязычных моделей (например, mBERT) используют 100 000–120 000. Для русского, татарского и таджикского можно начать с 50 000, если корпус смешанный.

3. **Настройте `min_frequency`.** Это отсекает редкие пары/подслова, что уменьшает словарь и улучшает обобщение. Обычно значение 2–5.

4. **Используйте подходящий пре-токенизатор.** Для языков с пробелами (русский, татарский, таджикский) подходит `Whitespace`, но для универсальности и работы с редкими символами лучше `ByteLevel` или `Metaspace`.

5. **Нормализуйте текст.** Это включает приведение к нижнему регистру, удаление диакритики, замену редких символов на `[UNK]`. Однако для татарского и таджикского важно сохранить дополнительные буквы (ә, ө, ү и др.), поэтому нельзя просто удалять диакритику. Нормализация Unicode может быть полезна.

6. **Проверьте декодирование.** Убедитесь, что после `encode` и `decode` текст восстанавливается корректно (с точностью до пробелов и нормализации). Это особенно важно для моделей, которые чувствительны к регистру.

7. **Сохраняйте токенизатор в JSON.** Это позволяет легко загружать его позже и делиться с другими.

## 10. Заключение

Библиотека HuggingFace Tokenizers предоставляет мощный и гибкий инструментарий для обучения и использования субсловных токенизаторов. В этой лекции мы рассмотрели, как обучить BPE, WordPiece и Unigram на собственном корпусе, как использовать предобученные токенизаторы из HuggingFace, и дали практические рекомендации. Мы также подробно остановились на обучении токенизаторов для русского, татарского и таджикского языков, учитывая их алфавитные и морфологические особенности. Выбор конкретного алгоритма зависит от задачи, языка и доступных вычислительных ресурсов, но в большинстве случаев все три метода дают приемлемое качество. Понимание их различий позволяет сделать осознанный выбор.

В следующей лекции мы перейдём к сравнительному анализу субсловных методов на реальном корпусе и рассмотрим влияние токенизации на качество downstream-задач.